# Hafta 6 · Etiketsiz Veride Yapı Aramak: Kümeleme ve Anomali

**SVY 5430 Biyomedikal Verilerin Analizi ve Yapay Zeka** · Fırat Üniversitesi Yazılım ve Bilişim Araştırma Enstitüsü

Bu defter kısa; ders 7. haftayla birlikte işleniyor. Üç bölüm: k-means ile hasta kümeleri, "kaç küme?" sorusu ve kümelerin gerçek olup
olmadığını sınamak, son olarak Isolation Forest ile anomali. Veri: Breast Cancer (569 hasta). Etiketi **saklıyoruz**; sonunda açıp bakacağız.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

ds = load_breast_cancer(as_frame=True)
X = ds.frame[ds.feature_names]; y_gizli = ds.frame["target"]     # 0 = malign, 1 = benign — şimdilik bakmıyoruz
Xs = StandardScaler().fit_transform(X)
Z = PCA(2).fit_transform(Xs)                                       # yalnızca çizim için
print(X.shape, "— etiketi kullanmıyoruz")

## 1 · k-means: k merkez seç, herkesi en yakına ata, merkezleri taşı, tekrarla

k-means'e kaç küme istediğinizi söylersiniz; o kadar küme bulur. Her zaman. Veride küme olmasa bile. Bu yüzden asıl iş, bulunan kümelerin
gerçek olup olmadığını sınamaktır.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, k in zip(ax, [2, 3, 4]):
    etiket = KMeans(k, n_init=10, random_state=0).fit_predict(Xs)
    a.scatter(Z[:, 0], Z[:, 1], c=etiket, cmap="viridis", s=12, alpha=.7)
    a.set_title(f"k = {k} · silhouette = {silhouette_score(Xs, etiket):.2f}"); a.set_xticks([]); a.set_yticks([])
plt.suptitle("Aynı veri, üç farklı k: hepsi 'küme' buldu", y=1.03); plt.tight_layout()

## 2 · Kaç küme? Silhouette ve kararlılık

**Silhouette** her hastanın kendi kümesine, komşu kümeye göre ne kadar yakın olduğunu ölçer (−1 ile 1; yüksek iyi).
**Kararlılık** ise şu soru: veriden rastgele bir yarısını alıp tekrar kümelesem aynı hastalar yine birlikte mi düşüyor?

In [ ]:
from sklearn.metrics import adjusted_rand_score
rng = np.random.default_rng(0)
print("k   silhouette   kararlılık (ARI, 20 alt-örneklem)")
for k in range(2, 7):
    sil = silhouette_score(Xs, KMeans(k, n_init=10, random_state=0).fit_predict(Xs))
    ref = KMeans(k, n_init=10, random_state=0).fit(Xs)
    aris = []
    for _ in range(20):
        idx = rng.choice(len(Xs), len(Xs) // 2, replace=False)
        alt = KMeans(k, n_init=10, random_state=int(rng.integers(1e6))).fit(Xs[idx])
        aris.append(adjusted_rand_score(ref.predict(Xs[idx]), alt.labels_))
    print(f"{k}   {sil:.3f}        {np.mean(aris):.3f}")

**Okuyun:** k = 2 hem silhouette hem kararlılıkta öne çıkıyor mu? Çıkıyorsa veri gerçekten iki parçalı olabilir.
Ama bu hâlâ iç kanıt. Dış kanıt, kümelemede kullanmadığımız bir bilgiyle örtüşme. Şimdi sakladığımız etiketi açıyoruz.

In [ ]:
etiket2 = KMeans(2, n_init=10, random_state=0).fit_predict(Xs)
tablo = pd.crosstab(pd.Series(etiket2, name="küme"), y_gizli.map({0: "malign", 1: "benign"}))
print(tablo, "\n")
print(f"Küme ile tanı uyumu (ARI): {adjusted_rand_score(y_gizli, etiket2):.3f}")

Kümeler tanıyla büyük ölçüde örtüşüyor; model tanıyı hiç görmeden, yalnızca hücre morfolojisinden iki grubu ayırdı. Gerçek alt-tiplendirme
çalışmasında "dış kanıt" bu tabloya benzer: sağkalım, tedavi yanıtı ya da kümelemede kullanılmamış bir klinik değişken.

> **Deneyin:** `KMeans` yerine `AgglomerativeClustering(2)` ve `GaussianMixture(2)` deneyin. Üç yöntem aynı hastaları mı ayırıyor?

## 3 · Anomali: ağaçla kolayca izole edilen nokta tuhaftır

Isolation Forest rastgele sorularla veriyi böler. Sıradan bir hasta çok soru sonra tek kalır; tuhaf bir hasta birkaç soruda. Skor, kaç soruda
tek kaldığıdır. Tuhaf = hatalı giriş mi, nadir ama gerçek vaka mı? Bunu yöntem değil, alan bilgisi söyler.

In [ ]:
from sklearn.ensemble import IsolationForest
iso = IsolationForest(n_estimators=300, contamination=0.03, random_state=0).fit(Xs)
skor = -iso.score_samples(Xs); anom = iso.predict(Xs) == -1
plt.figure(figsize=(6, 4.2))
plt.scatter(Z[~anom, 0], Z[~anom, 1], s=12, c="#bcc6d9", label="sıradan")
plt.scatter(Z[anom, 0], Z[anom, 1], s=40, c="#be3a5a", label=f"anomali ({anom.sum()})")
plt.legend(); plt.xticks([]); plt.yticks([]); plt.title("Isolation Forest: en tuhaf %3"); plt.tight_layout()
X.loc[anom, ["mean radius", "mean area", "worst area", "mean concave points"]].describe().loc[["min", "50%", "max"]].round(1)

**Okuyun:** Anomali olarak işaretlenen hastaların tümör alanı sıradan hastalardan ne kadar farklı? Bunlar hata mı, ileri evre vaka mı?

## 4 · Bugün ne gördük?

| Soru | Araç | Alışkanlık |
|---|---|---|
| Benzer hastalar kimler? | k-means | k'yi siz seçersiniz; her zaman k küme bulur |
| Kaç küme gerçek? | silhouette + kararlılık | İç kanıt yetmez |
| Kümeler anlamlı mı? | kümelemede kullanılmayan değişkenle çapraz tablo | Dış kanıt şart |
| Tuhaf hasta kim? | Isolation Forest | Hata mı nadir vaka mı: alan bilgisi karar verir |
